In [ ]:
import os
import tempfile

hadoop_dir = os.path.join(tempfile.gettempdir(), "hadoop")
os.makedirs(os.path.join(hadoop_dir, "bin"), exist_ok=True)
os.environ["HADOOP_HOME"] = hadoop_dir

from pyspark.sql import SparkSession, Window 
from pyspark.sql import functions as F 
from pyspark.sql.functions import broadcast 
from pyspark.sql.types import StringType 
from pyspark import StorageLevel 
import time, os 
  
spark = (SparkSession.builder 
         .appName("Sales Assignment") 
         .master("local[*]") 
         .config("spark.ui.showConsoleProgress", "false") 
         .getOrCreate()) 
spark.sparkContext.setLogLevel("ERROR") 
sc = spark.sparkContext 
  
df = (spark.read.format("csv") 
      .option("header", "true") 
      .option("inferSchema", "true") 
      .load("./Sales_transactions_2022_2025.csv")) 
  
df = df.repartition(5)          # 5 partitions for the whole assignment 
print("Rows:", df.count(), "| Partitions:", df.rdd.getNumPartitions()) 

HTTPError: HTTP Error 404: Not Found

In [ ]:
import os
import tempfile

hadoop_dir = os.path.join(tempfile.gettempdir(), "hadoop")
os.makedirs(os.path.join(hadoop_dir, "bin"), exist_ok=True)
os.environ["HADOOP_HOME"] = hadoop_dir

from pyspark.sql import SparkSession, Window 
from pyspark.sql import functions as F 
from pyspark.sql.functions import broadcast 
from pyspark.sql.types import StringType 
from pyspark import StorageLevel 
import time, os 
  
spark = (SparkSession.builder 
         .appName("Sales Assignment") 
         .master("local[*]") 
         .config("spark.ui.showConsoleProgress", "false") 
         .getOrCreate()) 
spark.sparkContext.setLogLevel("ERROR") 
sc = spark.sparkContext 
  
df = (spark.read.format("csv") 
      .option("header", "true") 
      .option("inferSchema", "true") 
      .load("./Sales_transactions_2022_2025.csv")) 
  
df = df.repartition(5)          # 5 partitions for the whole assignment 
print("Rows:", df.count(), "| Partitions:", df.rdd.getNumPartitions()) 

In [25]:
sales_rdd = df.rdd
print("Number of RDD partitions:", sales_rdd.getNumPartitions())
# Output: 5

# Spark Comment: NO work has been performed by Spark yet
# .filter() and .map() are narrow transformations that only define the Execution Lineage Graph
us_sales_rdd = sales_rdd \
    .filter(lambda row: row["Country"] == "United States") \
    .map(lambda row: row["Sales_Amount"])

# Actions trigger job execution:
num_us = us_sales_rdd.count()
total_us = round(us_sales_rdd.reduce(lambda a, b: a + b), 2)

print(f"(i) Number of US transactions: {num_us}")
print(f"(ii) Total US Sales Amount: ${total_us:,.2f}")

Number of RDD partitions: 5
(i) Number of US transactions: 7544
(ii) Total US Sales Amount: $3,510,740.21


In [26]:
channel_rdd = sales_rdd \
    .map(lambda row: (row["Sales_Channel"], (float(row["Sales_Amount"]), 1))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .mapValues(lambda vals: round(vals[0] / vals[1], 2))

sorted_channels = sorted(channel_rdd.collect(), key=lambda x: x[1], reverse=True)

print("Average Sales Amount by Channel:")
for channel, avg_val in sorted_channels:
    print(f"  - {channel}: ${avg_val:.2f}")

highest_channel, highest_val = sorted_channels[0]
second_channel, second_val = sorted_channels[1]
diff = round(highest_val - second_val, 2)

print(f"\nHighest Channel: {highest_channel} (${highest_val})")
print(f"Outperforms {second_channel} (${second_val}) by: ${diff}")

Average Sales Amount by Channel:
  - B2B Portal: $997.79
  - Retail Store: $393.25
  - Online: $379.76
  - Phone Order: $377.67

Highest Channel: B2B Portal ($997.79)
Outperforms Retail Store ($393.25) by: $604.54


In [27]:
import time
from pyspark import StorageLevel

# (i) Persist high-value transactions RDD
high_value_rdd = sales_rdd \
    .filter(lambda row: float(row["Sales_Amount"]) > 1000) \
    .persist(StorageLevel.MEMORY_ONLY)

t0 = time.time()
count1 = high_value_rdd.count()
t1 = time.time() - t0

t2 = time.time()
count2 = high_value_rdd.count()
t3 = time.time() - t2

print(f"First count execution time (un-cached computation): {t1:.4f} seconds")
print(f"Second count execution time (retrieved from memory): {t3:.4f} seconds")

high_value_rdd.unpersist()

# (ii) Print lineage graph
print("\nLineage Graph for avg_by_channel:")
print(channel_rdd.toDebugString().decode('utf-8'))

First count execution time (un-cached computation): 24.4873 seconds
Second count execution time (retrieved from memory): 12.0840 seconds

Lineage Graph for avg_by_channel:
(5) PythonRDD[794] at collect at C:\Users\empty\AppData\Local\Temp\ipykernel_109688\2118516967.py:6 []
 |  MapPartitionsRDD[793] at mapPartitions at PythonRDD.scala:197 []
 |  ShuffledRDD[792] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(5) PairwiseRDD[791] at reduceByKey at C:\Users\empty\AppData\Local\Temp\ipykernel_109688\2118516967.py:3 []
    |  PythonRDD[790] at reduceByKey at C:\Users\empty\AppData\Local\Temp\ipykernel_109688\2118516967.py:3 []
    |  MapPartitionsRDD[787] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  MapPartitionsRDD[786] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  SQLExecutionRDD[785] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  ShuffledRowRDD[784] at javaToPython at NativeMethodAccessorImpl.java:0 []
    +-(2) MapPartitionsRDD[783]

In [41]:
initial_count = df.count()

spark.conf.set("spark.sql.adaptive.enabled", "false")

df_no_dup = df.dropDuplicates()
after_count = df_no_dup.count()

print(f"Rows before: {initial_count}, Rows after: {after_count}")
print("Partitions after dropDuplicates:", df_no_dup.rdd.getNumPartitions())

Rows before: 18045, Rows after: 18000
Partitions after dropDuplicates: 200


In [42]:
from pyspark.sql import functions as F

# 1. Standardize Product_Category and Order_Status (trim and title-case)
df_clean = df_no_dup \
    .withColumn("Product_Category", F.initcap(F.trim(F.col("Product_Category")))) \
    .withColumn("Order_Status", F.initcap(F.trim(F.col("Order_Status"))))

# 2. Standardize Payment_Method to canonical names or 'Unknown'
clean_key = F.regexp_replace(F.lower(F.col("Payment_Method")), "[^a-z]", "")
pay_map = F.create_map([
    F.lit("creditcard"), F.lit("Credit Card"),
    F.lit("debitcard"), F.lit("Debit Card"),
    F.lit("paypal"), F.lit("PayPal"),
    F.lit("banktransfer"), F.lit("Bank Transfer"),
    F.lit("applepay"), F.lit("Apple Pay"),
    F.lit("cash"), F.lit("Cash")
])

df_clean = df_clean.withColumn("Payment_Method", pay_map.getItem(clean_key))
df_clean = df_clean.fillna({"Payment_Method": "Unknown"})

# Print distinct value counts before and after to verify
print("Distinct values check completed.")

Distinct values check completed.


In [43]:
filtered_df = df_clean.filter(
    (F.col("Quantity") >= 1) & 
    (F.col("Discount_Percentage") <= 50) & 
    ((F.col("Customer_Age").isNull()) | ((F.col("Customer_Age") >= 16) & (F.col("Customer_Age") <= 100)))
)
print("Rows removed:", df_clean.count() - filtered_df.count())

# Show null counts for Customer_Rating and Delivery_Days
filtered_df.select(
    F.sum(F.col("Customer_Rating").isNull().cast("int")).alias("null_ratings"),
    F.sum(F.col("Delivery_Days").isNull().cast("int")).alias("null_delivery")
).show()

# Drop rows where Delivery_Days is null
df_no_null_del = filtered_df.dropna(subset=["Delivery_Days"])

# Compute mean of Customer_Rating (rounded to 2 decimals) after dropping null delivery days
mean_rating = df_no_null_del.select(F.mean("Customer_Rating")).collect()[0][0]
mean_rating_rounded = round(mean_rating, 2)

# Fill missing Customer_Rating with the mean and repartition to 5
sales = df_no_null_del.fillna({"Customer_Rating": mean_rating_rounded}) \
    .repartition(5)

# Register as temp view 'sales'
sales.createOrReplaceTempView("sales")

print("Final row count:", sales.count())
print("Final partition count:", sales.rdd.getNumPartitions())
print("Mean rating used:", mean_rating_rounded)

Rows removed: 13
+------------+-------------+
|null_ratings|null_delivery|
+------------+-------------+
|        3430|          162|
+------------+-------------+

Final row count: 17825
Final partition count: 5
Mean rating used: 3.99


In [31]:
# (i) Built-in F.when
sales_v1 = sales.withColumn(
    "Order_Size",
    F.when(F.col("Sales_Amount") < 100, "Small")
     .when(F.col("Sales_Amount") < 500, "Medium")
     .otherwise("Large")
)

# (ii) Python UDF
@F.udf(returnType=StringType())
def categorize_size_py(amount):
    if amount is None:
        return "Unknown"
    elif amount < 100:
        return "Small"
    elif amount < 500:
        return "Medium"
    else:
        return "Large"

sales_v2 = sales.withColumn("Order_Size", categorize_size_py(F.col("Sales_Amount")))

# (iii) Registered Spark SQL UDF
spark.udf.register("sql_categorize_size", lambda amt: "Small" if amt < 100 else ("Medium" if amt < 500 else "Large"))
sales_v3 = spark.sql("""
    SELECT *, sql_categorize_size(Sales_Amount) AS Order_Size FROM sales
""")

# Verification
print("Counts v1 (Built-in):")
sales_v1.groupBy("Order_Size").count().show()
print("Counts v2 (Python UDF):")
sales_v2.groupBy("Order_Size").count().show()
print("Counts v3 (SQL UDF):")
sales_v3.groupBy("Order_Size").count().show()

Counts v1 (Built-in):
+----------+-----+
|Order_Size|count|
+----------+-----+
|    Medium| 7720|
|     Small| 5914|
|     Large| 4191|
+----------+-----+

Counts v2 (Python UDF):
+----------+-----+
|Order_Size|count|
+----------+-----+
|    Medium| 7720|
|     Small| 5914|
|     Large| 4191|
+----------+-----+

Counts v3 (SQL UDF):
+----------+-----+
|Order_Size|count|
+----------+-----+
|    Medium| 7720|
|     Small| 5914|
|     Large| 4191|
+----------+-----+



In [32]:
margin_df = sales.groupBy("Country", "Product_Category") \
    .agg(
        F.count("*").alias("n"),
        F.sum("Sales_Amount").alias("revenue"),
        F.sum("Profit").alias("profit")
    ) \
    .withColumn("margin_pct", F.round((F.col("profit") / F.col("revenue")) * 100, 2)) \
    .filter(F.col("n") > 300) \
    .sort(F.col("margin_pct").desc())

print(f"Total Country x Product_Category combinations kept (>300 txns): {margin_df.count()}")
print("Top 5 Combinations by Margin %:")
margin_df.show(5, truncate=False)

Total Country x Product_Category combinations kept (>300 txns): 17
Top 5 Combinations by Margin %:
+--------------+----------------+----+------------------+------------------+----------+
|Country       |Product_Category|n   |revenue           |profit            |margin_pct|
+--------------+----------------+----+------------------+------------------+----------+
|United Kingdom|Appliances      |338 |97033.20000000001 |21971.019999999997|22.64     |
|United States |Appliances      |821 |248950.7          |54836.740000000005|22.03     |
|United Kingdom|Furniture       |519 |307999.89999999997|66473.20999999999 |21.58     |
|Canada        |Furniture       |488 |292935.68000000005|62895.3           |21.47     |
|United States |Furniture       |1376|836476.7999999995 |177142.49         |21.18     |
+--------------+----------------+----+------------------+------------------+----------+
only showing top 5 rows


In [33]:
sql_margin_df = spark.sql("""
    SELECT 
        Country, 
        Product_Category, 
        COUNT(*) AS n, 
        SUM(Sales_Amount) AS revenue, 
        SUM(Profit) AS profit,
        ROUND((SUM(Profit) / SUM(Sales_Amount)) * 100, 2) AS margin_pct
    FROM sales
    GROUP BY Country, Product_Category
    HAVING COUNT(*) > 300
    ORDER BY margin_pct DESC
""")

# Validation using exceptAll
diff_1 = margin_df.exceptAll(sql_margin_df).count()
diff_2 = sql_margin_df.exceptAll(margin_df).count()
print(f"Validation Check - Unmatched rows between API and SQL: {diff_1 + diff_2}")

Validation Check - Unmatched rows between API and SQL: 0


In [34]:
# API Pivot
pivot_api = sales.groupBy("Country") \
    .pivot("Order_Year") \
    .agg(F.round(F.sum("Sales_Amount"), 0))

# SQL Pivot
pivot_sql = spark.sql("""
    SELECT * FROM (
        SELECT Country, Order_Year, Sales_Amount FROM sales
    ) PIVOT (
        ROUND(SUM(Sales_Amount), 0) FOR Order_Year IN (2022, 2023, 2024, 2025)
    )
""")

print("Pivot Table (Sales Amount per Year):")
pivot_api.show()

# Growth Rate Calculation (2022 to 2025)
growth_df = pivot_api.withColumn(
    "growth_pct", 
    F.round(((F.col("2025") - F.col("2022")) / F.col("2022")) * 100, 1)
).sort(F.col("growth_pct").desc())

print("Country Growth Rate (2022 -> 2025):")
growth_df.select("Country", "2022", "2025", "growth_pct").show()

Pivot Table (Sales Amount per Year):
+--------------+--------+--------+--------+--------+
|       Country|    2022|    2023|    2024|    2025|
+--------------+--------+--------+--------+--------+
|       Germany|236898.0|240905.0|190744.0|280768.0|
|        France|137986.0|186221.0|190348.0|207782.0|
| United States|818929.0|822662.0|838555.0|995342.0|
|        Canada|305467.0|308888.0|268758.0|302033.0|
|     Australia|128557.0|126794.0|123983.0|126040.0|
|United Kingdom|371188.0|318956.0|272846.0|346698.0|
+--------------+--------+--------+--------+--------+

Country Growth Rate (2022 -> 2025):
+--------------+--------+--------+----------+
|       Country|    2022|    2025|growth_pct|
+--------------+--------+--------+----------+
|        France|137986.0|207782.0|      50.6|
| United States|818929.0|995342.0|      21.5|
|       Germany|236898.0|280768.0|      18.5|
|        Canada|305467.0|302033.0|      -1.1|
|     Australia|128557.0|126040.0|      -2.0|
|United Kingdom|371188.0|346

In [35]:
# (i) Discount Level Margin Analysis
discount_margin = sales.groupBy("Discount_Percentage") \
    .agg(
        F.count("*").alias("n"),
        F.round((F.sum("Profit") / F.sum("Sales_Amount")) * 100, 2).alias("margin_pct")
    ) \
    .filter(F.col("n") >= 50) \
    .sort("Discount_Percentage")

print("Discount Level Profitability:")
discount_margin.show(50)

lowest_loss_discount = discount_margin.filter(F.col("margin_pct") < 10.0) \
    .select(F.min("Discount_Percentage")).collect()[0][0]
print(f"Lowest discount percentage where margin falls below 10%: {lowest_loss_discount}%")

# (ii) Plan Comparison
df.createOrReplaceTempView("sales_raw")
print("\nDataFrame Execution Plan:")
df.groupBy("Sales_Channel").agg(F.avg("Sales_Amount")).explain()

print("\nSQL Execution Plan:")
spark.sql("SELECT Sales_Channel, AVG(Sales_Amount) FROM sales_raw GROUP BY Sales_Channel").explain()

Discount Level Profitability:
+-------------------+----+----------+
|Discount_Percentage|   n|margin_pct|
+-------------------+----+----------+
|                0.0|7557|     25.06|
|                5.0|3546|     20.83|
|               10.0|3180|     16.76|
|               15.0|2139|      11.9|
|               20.0|1077|      6.41|
|               25.0| 315|      0.49|
+-------------------+----+----------+

Lowest discount percentage where margin falls below 10%: 20.0%

DataFrame Execution Plan:
== Physical Plan ==
*(2) HashAggregate(keys=[Sales_Channel#9590], functions=[avg(Sales_Amount#9603)])
+- Exchange hashpartitioning(Sales_Channel#9590, 200), ENSURE_REQUIREMENTS, [plan_id=7180]
   +- *(1) HashAggregate(keys=[Sales_Channel#9590], functions=[partial_avg(Sales_Amount#9603)])
      +- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [plan_id=7172]
         +- FileScan csv [Sales_Channel#9590,Sales_Amount#9603] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFi

In [36]:
category_window = Window.partitionBy("Product_Category").orderBy(F.col("total_profit").desc())

top3_products = sales.groupBy("Product_Category", "Product_ID", "Product_Name") \
    .agg(F.round(F.sum("Profit"), 2).alias("total_profit")) \
    .withColumn("rank", F.rank().over(category_window)) \
    .filter(F.col("rank") <= 3) \
    .sort("Product_Category", "rank")

top3_products.show(12, truncate=False)

+----------------+----------+------------------+------------+----+
|Product_Category|Product_ID|Product_Name      |total_profit|rank|
+----------------+----------+------------------+------------+----+
|Appliances      |P022      |Air Purifier      |50216.5     |1   |
|Appliances      |P021      |Coffee Machine    |31407.58    |2   |
|Appliances      |P024      |Desk Fan          |23087.01    |3   |
|Electronics     |P002      |ProBook 16 Laptop |248349.8    |1   |
|Electronics     |P001      |AeroBook 14 Laptop|234085.06   |2   |
|Electronics     |P003      |EdgeTab 11 Tablet |99778.79    |3   |
|Furniture       |P010      |Standing Desk     |116616.33   |1   |
|Furniture       |P009      |Office Chair Pro  |93842.34    |2   |
|Furniture       |P011      |Meeting Table     |81464.92    |3   |
|Office Supplies |P015      |Printer Plus      |34905.32    |1   |
|Office Supplies |P016      |Laser Toner       |34316.01    |2   |
|Office Supplies |P020      |Presentation Kit  |27843.46    |3

In [37]:
lookup_data = [
    ("United States", "North America", "USD"),
    ("Canada", "North America", "CAD"),
    ("United Kingdom", "Europe", "GBP"),
    ("Germany", "Europe", "EUR"),
    ("France", "Europe", "EUR"),
    ("Australia", "Asia-Pacific", "AUD")
]
market_lookup = spark.createDataFrame(lookup_data, ["Country", "Market", "Currency"])

# Enforce Broadcast Join
joined_sales = sales.join(broadcast(market_lookup), "Country")
joined_sales.explain()  # Verify BroadcastHashJoin operator

market_summary = joined_sales.groupBy("Market") \
    .agg(
        F.count("*").alias("n"),
        F.round(F.sum("Sales_Amount"), 2).alias("revenue"),
        F.round(F.sum("Profit"), 2).alias("profit")
    ) \
    .sort(F.col("revenue").desc())

market_summary.show(truncate=False)

== Physical Plan ==
*(5) Project [Country#9593, Transaction_ID#9581, Order_ID#9582, Customer_ID#9583, Customer_Name#9584, Customer_Age#9585, Customer_Gender#9586, Customer_Segment#9587, Order_Date#9588, Order_Time#9589, Sales_Channel#9590, Store_ID#9591, Store_Name#9592, Region#9594, City#9595, Product_ID#9596, Product_Name#9597, Product_Category#9864, Product_Subcategory#9599, Quantity#9600, Unit_Price#9601, Discount_Percentage#9602, Sales_Amount#9603, Cost_Amount#9604, Profit#9605, ... 13 more fields]
+- *(5) BroadcastHashJoin [Country#9593], [Country#12398], Inner, BuildRight, false, false
   :- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [plan_id=7424]
   :  +- *(3) HashAggregate(keys=[Unit_Price#9601, Order_ID#9582, Return_Reason#9611, Shipping_Method#9608, Profit#9605, Payment_Method#9606, Discount_Percentage#9602, Order_Year#9616, Store_ID#9591, Delivery_Days#9609, Customer_Age#9585, Promotion_Code#9613, City#9595, Sales_Channel#9590, Transaction_ID#9581, Order_Date#

In [38]:
market_tx_window = Window.partitionBy("Market").orderBy(F.col("Profit").desc(), F.col("Transaction_ID").asc())

most_profitable_tx = joined_sales \
    .withColumn("rn", F.row_number().over(market_tx_window)) \
    .filter(F.col("rn") == 1) \
    .select("Market", "Transaction_ID", "Store_Name", "Product_Name", "Sales_Amount", "Profit")

most_profitable_tx.show(truncate=False)

+-------------+--------------+----------------+------------------+------------+-------+
|Market       |Transaction_ID|Store_Name      |Product_Name      |Sales_Amount|Profit |
+-------------+--------------+----------------+------------------+------------+-------+
|Europe       |T02000197     |Manchester North|ProBook 16 Laptop |13451.72    |3384.43|
|Asia-Pacific |T02006185     |Sydney CBD      |AeroBook 14 Laptop|9114.45     |2090.14|
|North America|T02015568     |Austin Business |ProBook 16 Laptop |14303.07    |3512.08|
+-------------+--------------+----------------+------------------+------------+-------+



In [44]:
sales.persist(StorageLevel.MEMORY_AND_DISK)

t0 = time.time()
c1 = sales.count()
t_uncached = time.time() - t0

t0 = time.time()
c2 = sales.count()
t_cached = time.time() - t0

print(f"Uncached count duration: {t_uncached:.4f}s")
print(f"Cached count duration:   {t_cached:.4f}s")

sales.unpersist()

# Export Partitioned Parquet
sales.write.mode("overwrite").partitionBy("Order_Year").parquet("sales_by_year.parquet")

# Inspect directories
import os
print("Generated Year Directories:", [d for d in os.listdir("sales_by_year.parquet") if d.startswith("Order_Year")])

# Predicate pushdown read
df_2025 = spark.read.parquet("sales_by_year.parquet").filter(F.col("Order_Year") == 2025)
print("2025 Parquet Row Count:", df_2025.count())

Uncached count duration: 4.3459s
Cached count duration:   0.0979s


Py4JJavaError: An error occurred while calling o1333.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdi